## 对话状态

LLM 是无状态的——它们在调用之间没有记忆。您可以通过随每个请求发送完整的消息列表来管理对话历史记录。理解这一点是基础

资源：

1、OpenAI 聊天完成指南，管理对话（官方，免费）

链接：<https://platform.openai.com/docs/guides/conversation-state>

关于消息数组如何工作以及如何管理多轮对话的规范解释

2、Anthropic Messages API 文档（官方，免费）

链接：<https://platform.claude.com/docs/zh-CN/build-with-claude/working-with-messages#>

人择的等价物。相同的概念，值得阅读以了解它们有何不同

**关注点：** 消息数组结构、为什么要附加用户消息和助理消息、上下文窗口限制以及超出限制时会发生什么，以及基本截断策略（删除最旧的消息、总结历史记录）

**实践项目：** 在终端中构建一个简单的多回合聊天机器人。每轮都会附加到消息列表中。添加 /reset 命令以清除历史记录，并在每次交换后打印当前代币计数

## 手动管理对话状态

每个文本生成请求都是独立且无状态的，通过交替使用user和assistant消息，您可以在单次模型请求中捕捉对话的先前状态。


In [ ]:
import OpenAI from 'openai'

const openai = new OpenAI()


In [ ]:
const response = await openai.responses.create({
  model: 'gpt-5.4-nano',
  input: [
    { role: 'user', content: 'knock knock.' },
    { role: 'assistant', content: "Who's there?" },
    { role: 'user', content: 'Orange.' },
  ],
})

console.log(response.output_text)


In [ ]:
let history: OpenAI.Responses.ResponseInput = [
  {
    role: 'user',
    content: 'tell me a joke',
  },
]

const response1 = await openai.responses.create({
  model: 'gpt-5.4-nano',
  input: history,
})

console.log(response1.output_text)

// Add the response to the history
history = [
  ...history,
  ...response1.output.map((el) => {
    // TODO: Remove this step
    delete el.id
    return el
  }),
]

history.push({
  role: 'user',
  content: 'tell me another',
})

const secondResponse = await openai.responses.create({
  model: 'gpt-5.4-nano',
  input: history,
})

console.log(secondResponse.output_text)
console.log(history)


## 官方能力管理对话状态

### 使用对话API

对话API与响应API协同工作，将对话状态持久化为一个具有自身持久标识符的长期运行对象。创建对话对象后，您可以在不同会话、设备或任务中持续使用它。

> Conversations API 只有使用OpenAI官方的API才有用，接入的第三方的API不可用

In [ ]:
// 创建对话
const conversation = await openai.conversations.create()

console.log('Conversation ID:', conversation.id)


In [ ]:
const response2 = await openai.responses.create({
  model: 'gpt-5.4-nano',
  conversation: conversation.id,
  input: [
    { role: 'user', content: 'What is my name?' },
  ],
})

console.log('Second Response:', response2.output_text)


### 从之前的响应传递上下文

另一种管理对话状态的方式是通过previous_response_id参数在生成的响应之间共享上下文。该参数允许您链接响应并创建线程式对话。

In [ ]:
const response3 = await openai.responses.create({
  model: 'gpt-5.4-nano',
  input: '给我讲个笑话',
})

console.log(response3.output_text)

const response4 = await openai.responses.create({
  model: 'gpt-5.4-nano',
  previous_response_id: response3.id,
  input: [{ 'role': 'user', 'content': '解释一下为什么有趣' }],
})

console.log(response4.output_text)
